# CLV-Conditioned Multi-Embedding LightGCN

H&M 2년 또는 Dunnhumby의 M2 seed 42 validation screening용 노트북입니다. 기본 상태에서는 test·holdout 정답을 만들지 않으며, 고비용 학습 셀은 설정 검토 후 명시적으로 승인해야 실행됩니다.

## 1. Setup — Drive와 최신 구현 브랜치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, sys

REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
BRANCH = 'feat/clv-conditioned-moe'
REPO_DIR = Path('/content/clv-m2-lightgcn-runner')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
sys.path.insert(0, str(REPO_DIR))
print('repo:', REPO_DIR, '| branch:', BRANCH)

## 2. Parameters

`DATASET`만 `dunnhumby` 또는 `hm`으로 바꾸세요. 두 경우 모두 동일한 모형 원리·seed 42·validation-only 규칙을 사용합니다.

In [ ]:
import json, torch
from lightgcn_clv_moe import configure_moe_run, preflight_summary, run_experiment

DATASET = 'dunnhumby'  # 'dunnhumby' or 'hm'
RESULT_ROOT = Path('/content/drive/MyDrive/논문/data')
cfg = configure_moe_run(
    DATASET,
    seed_list=(42,),
    eval_test=False,
    eval_holdout=False,
    out_dir=str(RESULT_ROOT / f'results_clv_moe_{DATASET}'),
    m1_checkpoint_dir=str(RESULT_ROOT / f'results_v3_{DATASET}'),
)
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'

## 3. Preflight — 이 출력까지는 학습하지 않음

dataset, seed, split 보호, M1 경로, 결과 경로, λ grid와 M2/M3/M4 경계를 한 번에 검토합니다.

In [ ]:
summary = preflight_summary(cfg)
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert summary['seed_list'] == [42]
assert summary['eval_test'] is False and summary['eval_holdout'] is False
assert summary['graph_mode'] == 'binary' and summary['loss_mode'] == 'plain'
assert summary['expert_count'] == 3

## 4. High-cost approval

위 preflight 전체를 검토한 뒤에만 아래 값을 `True`로 변경하세요. `False`인 동안 실제 데이터 준비·M1 로드·encoder/MoE 학습은 시작되지 않습니다.

In [ ]:
ACKNOWLEDGE_HIGH_COST = False
assert ACKNOWLEDGE_HIGH_COST, '설정 검토 후 ACKNOWLEDGE_HIGH_COST=True로 바꾸세요.'
result_df = run_experiment(cfg)

## 5. Validation results

In [ ]:
display_columns = [
    'seed', 'model_id', 'split', 'lambda', 'recall@10', 'ndcg@10',
    'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50', 'revenue@10',
    'arp@10', 'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
    'value_alignment', 'gate_entropy_mean', 'residual_to_base_score_std'
]
available = [column for column in display_columns if column in result_df.columns]
display(result_df[available].sort_values(['model_id', 'split', 'lambda']))
print('결과 파일:')
for path in sorted(Path(cfg.out_dir).glob('clv_moe_*')):
    print(' -', path)

## 6. Interpretation guardrails

- 선택 λ가 0이면 안전 fallback이며 제안 M2 성공이 아닙니다.
- `revenue`는 가격·구매금액 가중 적중값이지 실제 증분매출이나 CLV가 아닙니다.
- seed 42 validation screening 결과로 test 설정을 바꾸지 않습니다.
- 주 모형이 양의 λ로 성공할 때만 대조군이 자동 실행됩니다. 두 데이터셋 모두 성공하기 전에는 다중 seed·test로 진행하지 않습니다.